In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Part_6_window").getOrCreate()
print("spark session ready")

spark session ready


In [2]:
products_data = [
(101,"Rice Bag","Groceries","Hyderabad",1200,50),
(102,"Wheat Flour","Groceries","Bengaluru",900,80),
(103,"Sunflower Oil","Groceries","Mumbai",1800,40),
(104,"Milk Pack","Dairy","Chennai",60,200),
(105,"Cheese Block","Dairy","Delhi",450,70),
(106,"Soap","Personal Care","Kolkata",120,300),
(107,"Shampoo","Personal Care","Pune",320,150),
(108,"Toothpaste","Personal Care","Ahmedabad",90,250),
(109,"Notebook","Stationery","Hyderabad",75,500),
(110,"Pen Pack","Stationery","Mumbai",110,400),
(111,"LED TV","Electronics","Delhi",45000,15),
(112,"Refrigerator","Electronics","Chennai",38000,10),
(113,"Washing Machine","Electronics","Bengaluru",29000,12),
(114,"Mobile Phone","Electronics","Hyderabad",25000,35),
(115,"Laptop","Electronics","Pune",62000,18),
(116,"Air Conditioner","Electronics","Mumbai",42000,9),
(117,"Mixer Grinder","Home Appliances","Kolkata",3500,45),
(118,"Water Purifier","Home Appliances","Delhi",12000,20),
(119,"Ceiling Fan","Home Appliances","Ahmedabad",2800,60),
(120,"Gas Stove","Home Appliances","Chennai",5500,25)
]
products_columns = [
"product_id",
"product_name",
"category",
"warehouse_city",
"price",
"stock_quantity"
]
products_df = spark.createDataFrame(products_data, products_columns)

In [3]:
suppliers_data = [
(201,"Reddy Traders","Hyderabad","Groceries"),
(202,"Fresh Dairy Ltd","Chennai","Dairy"),
(203,"CarePlus Suppliers","Mumbai","Personal Care"),
(204,"Elite Electronics","Delhi","Electronics"),
(205,"OfficeKart","Bengaluru","Stationery"),
(206,"HomeNeeds Pvt Ltd","Pune","Home Appliances"),
(207,"National Grocers","Ahmedabad","Groceries"),
(208,"Smart Electronics","Kolkata","Electronics"),
(209,"Daily Essentials","Hyderabad","Personal Care"),
(210,"Kitchen World","Chennai","Home Appliances")
]
suppliers_columns = [
"supplier_id",
"supplier_name",
"supplier_city",
"specialization"
]
suppliers_df = spark.createDataFrame(suppliers_data, suppliers_columns)

In [4]:
orders_data = [
(301,101,201,"2024-04-01",20,"Delivered"),
(302,102,201,"2024-04-01",35,"Delivered"),
(303,111,204,"2024-04-02",2,"Delivered"),
(304,114,208,"2024-04-02",5,"Pending"),
(305,115,204,"2024-04-03",3,"Delivered"),
(306,104,202,"2024-04-03",50,"Delivered"),
(307,105,202,"2024-04-04",18,"Cancelled"),
(308,117,206,"2024-04-04",7,"Delivered"),
(309,118,210,"2024-04-05",4,"Pending"),
(310,119,206,"2024-04-05",12,"Delivered"),
(311,120,210,"2024-04-06",6,"Delivered"),
(312,113,204,"2024-04-06",4,"Delivered"),
(313,116,208,"2024-04-07",2,"Pending"),
(314,109,205,"2024-04-07",80,"Delivered"),
(315,110,205,"2024-04-08",120,"Delivered"),
(316,106,203,"2024-04-08",60,"Cancelled"),
(317,107,209,"2024-04-09",25,"Delivered"),
(318,108,203,"2024-04-09",40,"Delivered"),
(319,112,208,"2024-04-10",2,"Pending"),
(320,101,207,"2024-04-10",15,"Delivered")
]
orders_columns = [
"order_id",
"product_id",
"supplier_id",
"order_date",
"quantity",
"order_status"
]
orders_df = spark.createDataFrame(orders_data, orders_columns)

In [5]:
payments_data = [
(401,301,24000,"UPI","Paid"),
(402,302,31500,"Credit Card","Paid"),
(403,303,90000,"Bank Transfer","Paid"),
(404,304,125000,"UPI","Pending"),
(405,305,186000,"Bank Transfer","Paid"),
(406,306,3000,"Cash","Paid"),
(407,307,8100,"UPI","Cancelled"),
(408,308,24500,"Debit Card","Paid"),
(409,309,48000,"UPI","Pending"),
(410,310,33600,"Cash","Paid"),
(411,311,33000,"Credit Card","Paid"),
(412,312,116000,"Bank Transfer","Paid"),
(413,313,84000,"UPI","Pending"),
(414,314,6000,"Cash","Paid"),
(415,315,13200,"UPI","Paid"),
(416,316,7200,"Cash","Cancelled"),
(417,317,8000,"UPI","Paid"),
(418,318,3600,"Debit Card","Paid"),
(419,319,76000,"Bank Transfer","Pending"),
(420,320,18000,"UPI","Paid")
]
payments_columns = [
"payment_id",
"order_id",
"bill_amount",
"payment_mode",
"payment_status"
]
payments_df = spark.createDataFrame(payments_data, payments_columns)

In [7]:
from pyspark.sql.functions import col, rank, dense_rank, row_number, sum, count, max, min, avg, when, upper, udf, to_date
from pyspark.sql.window import Window
#Exercise 51
final_df = orders_df.join(products_df, "product_id") \
                    .join(suppliers_df, "supplier_id") \
                    .join(payments_df, "order_id")
product_rev= final_df.groupBy("category", "product_name")\
                     .agg(sum("bill_amount").alias("total_revenue"))
window = Window.orderBy(col("total_revenue").desc())
product_rev.withColumn("product_rank", rank().over(window)).show()


+---------------+---------------+-------------+------------+
|       category|   product_name|total_revenue|product_rank|
+---------------+---------------+-------------+------------+
|    Electronics|         Laptop|       186000|           1|
|    Electronics|   Mobile Phone|       125000|           2|
|    Electronics|Washing Machine|       116000|           3|
|    Electronics|         LED TV|        90000|           4|
|    Electronics|Air Conditioner|        84000|           5|
|    Electronics|   Refrigerator|        76000|           6|
|Home Appliances| Water Purifier|        48000|           7|
|      Groceries|       Rice Bag|        42000|           8|
|Home Appliances|    Ceiling Fan|        33600|           9|
|Home Appliances|      Gas Stove|        33000|          10|
|      Groceries|    Wheat Flour|        31500|          11|
|Home Appliances|  Mixer Grinder|        24500|          12|
|     Stationery|       Pen Pack|        13200|          13|
|          Dairy|   Chee

In [10]:
#Exercise 52
supp_rev = final_df.groupby("supplier_city","supplier_name").agg(sum("bill_amount").alias("total_revenue"))

window= Window.orderBy(col("total_revenue").desc())
supp_rev.withColumn("supplier_rank", rank().over(window)).show()

+-------------+------------------+-------------+-------------+
|supplier_city|     supplier_name|total_revenue|supplier_rank|
+-------------+------------------+-------------+-------------+
|        Delhi| Elite Electronics|       392000|            1|
|      Kolkata| Smart Electronics|       285000|            2|
|      Chennai|     Kitchen World|        81000|            3|
|         Pune| HomeNeeds Pvt Ltd|        58100|            4|
|    Hyderabad|     Reddy Traders|        55500|            5|
|    Bengaluru|        OfficeKart|        19200|            6|
|    Ahmedabad|  National Grocers|        18000|            7|
|      Chennai|   Fresh Dairy Ltd|        11100|            8|
|       Mumbai|CarePlus Suppliers|        10800|            9|
|    Hyderabad|  Daily Essentials|         8000|           10|
+-------------+------------------+-------------+-------------+



In [16]:
#Exercise 53
window = Window.partitionBy("category").orderBy(col("total_revenue").desc())
product_rev.withColumn("product_rn", row_number().over(window)).filter(col("product_rn")==1).show()

+---------------+--------------+-------------+----------+
|       category|  product_name|total_revenue|product_rn|
+---------------+--------------+-------------+----------+
|          Dairy|  Cheese Block|         8100|         1|
|    Electronics|        Laptop|       186000|         1|
|      Groceries|      Rice Bag|        42000|         1|
|Home Appliances|Water Purifier|        48000|         1|
|  Personal Care|       Shampoo|         8000|         1|
|     Stationery|      Pen Pack|        13200|         1|
+---------------+--------------+-------------+----------+



In [18]:
#Exercise 54
window = Window.partitionBy("supplier_name").orderBy(col("bill_amount").desc())

final_df.select("supplier_name", "bill_amount").withColumn("supp_dense_rank", dense_rank().over(window)).show()

+------------------+-----------+---------------+
|     supplier_name|bill_amount|supp_dense_rank|
+------------------+-----------+---------------+
|CarePlus Suppliers|       7200|              1|
|CarePlus Suppliers|       3600|              2|
|  Daily Essentials|       8000|              1|
| Elite Electronics|     186000|              1|
| Elite Electronics|     116000|              2|
| Elite Electronics|      90000|              3|
|   Fresh Dairy Ltd|       8100|              1|
|   Fresh Dairy Ltd|       3000|              2|
| HomeNeeds Pvt Ltd|      33600|              1|
| HomeNeeds Pvt Ltd|      24500|              2|
|     Kitchen World|      48000|              1|
|     Kitchen World|      33000|              2|
|  National Grocers|      18000|              1|
|        OfficeKart|      13200|              1|
|        OfficeKart|       6000|              2|
|     Reddy Traders|      31500|              1|
|     Reddy Traders|      24000|              2|
| Smart Electronics|

In [20]:
#Exercise 55
window = Window.orderBy(col("total_revenue").desc())
product_rev.withColumn("top_suppliers", rank().over(window)).filter(col("top_suppliers")<=2).show()

+-----------+------------+-------------+-------------+
|   category|product_name|total_revenue|top_suppliers|
+-----------+------------+-------------+-------------+
|Electronics|      Laptop|       186000|            1|
|Electronics|Mobile Phone|       125000|            2|
+-----------+------------+-------------+-------------+



In [21]:
#Exercise 56
daily_rev= final_df.groupby("order_date")\
           .agg(sum("bill_amount").alias("daily_revenue"))\
           .orderBy("order_date")
window=  Window.orderBy(col("order_date")).rowsBetween(Window.unboundedPreceding, Window.currentRow)
daily_rev.withColumn("running_revenue", sum("daily_revenue").over(window)).show()

+----------+-------------+---------------+
|order_date|daily_revenue|running_revenue|
+----------+-------------+---------------+
|2024-04-01|        55500|          55500|
|2024-04-02|       215000|         270500|
|2024-04-03|       189000|         459500|
|2024-04-04|        32600|         492100|
|2024-04-05|        81600|         573700|
|2024-04-06|       149000|         722700|
|2024-04-07|        90000|         812700|
|2024-04-08|        20400|         833100|
|2024-04-09|        11600|         844700|
|2024-04-10|        94000|         938700|
+----------+-------------+---------------+



In [23]:
#Exercise 57
supplier_rev= final_df.groupby("supplier_name", "order_date")\
                      .agg(sum("bill_amount").alias("daily_revenue"))\
                      .orderBy("supplier_name","order_date")
window= Window.partitionBy("supplier_name").orderBy("order_date")\
.rowsBetween(Window.unboundedPreceding, Window.currentRow)

supplier_rev.withColumn("running_revenue", sum("daily_revenue").over(window)).show()

+------------------+----------+-------------+---------------+
|     supplier_name|order_date|daily_revenue|running_revenue|
+------------------+----------+-------------+---------------+
|CarePlus Suppliers|2024-04-08|         7200|           7200|
|CarePlus Suppliers|2024-04-09|         3600|          10800|
|  Daily Essentials|2024-04-09|         8000|           8000|
| Elite Electronics|2024-04-02|        90000|          90000|
| Elite Electronics|2024-04-03|       186000|         276000|
| Elite Electronics|2024-04-06|       116000|         392000|
|   Fresh Dairy Ltd|2024-04-03|         3000|           3000|
|   Fresh Dairy Ltd|2024-04-04|         8100|          11100|
| HomeNeeds Pvt Ltd|2024-04-04|        24500|          24500|
| HomeNeeds Pvt Ltd|2024-04-05|        33600|          58100|
|     Kitchen World|2024-04-05|        48000|          48000|
|     Kitchen World|2024-04-06|        33000|          81000|
|  National Grocers|2024-04-10|        18000|          18000|
|       

In [24]:
#Exercise 58
city_rev= final_df.groupby("warehouse_city").agg(sum("bill_amount").alias("total_revenue"))

window = Window.orderBy(col("total_revenue").desc())
city_rev.withColumn("city_rank", rank().over(window)).show()

+--------------+-------------+---------+
|warehouse_city|total_revenue|city_rank|
+--------------+-------------+---------+
|          Pune|       194000|        1|
|     Hyderabad|       173000|        2|
|     Bengaluru|       147500|        3|
|         Delhi|       146100|        4|
|       Chennai|       112000|        5|
|        Mumbai|        97200|        6|
|     Ahmedabad|        37200|        7|
|       Kolkata|        31700|        8|
+--------------+-------------+---------+



In [25]:
#Exercise 59
cat_rev= final_df.groupby("category").agg(sum("bill_amount").alias("total_revenue"))

window = Window.orderBy(col("total_revenue").desc())
cat_rev.withColumn("category_rank", rank().over(window)).show()

+---------------+-------------+-------------+
|       category|total_revenue|category_rank|
+---------------+-------------+-------------+
|    Electronics|       677000|            1|
|Home Appliances|       139100|            2|
|      Groceries|        73500|            3|
|     Stationery|        19200|            4|
|  Personal Care|        18800|            5|
|          Dairy|        11100|            6|
+---------------+-------------+-------------+



In [29]:
#Exercise 60
window = Window.partitionBy("payment_mode").orderBy(col("bill_amount").desc())

final_df.withColumn("highest_bill", rank().over(window)).filter(col("highest_bill")==1).select("payment_mode", "order_id", "bill_amount").show()

+-------------+--------+-----------+
| payment_mode|order_id|bill_amount|
+-------------+--------+-----------+
|Bank Transfer|     305|     186000|
|         Cash|     310|      33600|
|  Credit Card|     311|      33000|
|   Debit Card|     308|      24500|
|          UPI|     304|     125000|
+-------------+--------+-----------+

